In [1]:
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data_1000.csv")

In [5]:
df["age"] = pd.to_numeric(df["age"], errors = "coerce")
df["joining_date"] = pd.to_datetime(df["joining_date"], errors = "coerce")

/tmp/ipykernel_4292/2420527129.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["joining_date"] = pd.to_datetime(df["joining_date"], errors = "coerce")


In [6]:
print("\n====== DATA OVERVIEW =======")
print("Rows, Columns:", df.shape)


====== DATA OVERVIEW =======
Rows, Columns: (1020, 6)


In [7]:
missing = df.isnull().sum()
missing_percent = (missing/len(df))*100

In [8]:
duplicates = df.duplicated().sum()


In [35]:
numeric_df = df.select_dtypes(include=['number'])

Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
IQR = Q3 - Q1

outliers = ((numeric_df < (Q1 - 1.5 * IQR)) | (numeric_df > (Q3 + 1.5 * IQR)))
outlier_count = outliers.sum()

In [14]:
score = 100
score -= int(missing_percent.mean())
score -= duplicates
score -= int(outlier_count.sum()/10)
score = max(0,score)

In [16]:
grades = {}

for col in df.columns:
  miss = missing_percent[col]
  if miss == 0:
    grades[col] = "A"
  elif miss < 10:
    grades[col] = "B"
  elif miss < 30:
    grades[col] = "C"
  else:
    grades[col] = "D"

In [19]:
suggestions = []

if missing.sum() > 0:
  suggestions.append("Fill missing values using median or mode")

if duplicates > 0:
  suggestions.append("REmove duplicate rows")

suggestions.append("Convert incorrect datatypes")
suggestions.append("Handle outliers using IQR or capping")

In [21]:
clean_df = df.copy()
clean_df = clean_df.drop_duplicates()
clean_df = clean_df.fillna(clean_df.median(numeric_only = True))

In [22]:
print("\n===== DATA QUALITY REPORT =====")
print("Quality Score:", score, "/ 100")

print("\nMissing Values (%)")
print(missing_percent)




===== DATA QUALITY REPORT =====
Quality Score: 54 / 100

Missing Values (%)
id               0.000000
name             0.000000
age             68.333333
salary          32.941176
department       8.725490
joining_date    49.313725
dtype: float64


In [23]:
print("\nDuplicates:", duplicates)

print("\nOutliers:")
print(outlier_count)





Duplicates: 20

Outliers:
id        0
age       0
salary    0
dtype: int64


In [24]:
print("\nColumn Grades:")
for col, grade in grades.items():
    print(f"{col}: {grade}")

print("\nSuggestions:")
for s in suggestions:
    print("-", s)

print("\nBefore Cleaning:", df.shape)
print("After Cleaning:", clean_df.shape)


Column Grades:
id: A
name: A
age: D
salary: D
department: B
joining_date: D

Suggestions:
- Fill missing values using median or mode
- REmove duplicate rows
- Convert incorrect datatypes
- Handle outliers using IQR or capping

Before Cleaning: (1020, 6)
After Cleaning: (1000, 6)


In [26]:
with open("final_report.txt", "w") as f:
    f.write("DATA QUALITY REPORT\n\n")
    f.write(f"Quality Score: {score}/100\n\n")
    f.write("Missing Values (%):\n")
    f.write(str(missing_percent))
    f.write("\n\nDuplicates: " + str(duplicates))
    f.write("\n\nColumn Grades:\n")
    for col, grade in grades.items():
        f.write(f"{col}: {grade}\n")

print("\nReport saved as final_report.txt")


Report saved as final_report.txt


In [28]:
clean_df.to_csv("cleaned_data.csv", index=False)

In [29]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.3 MB/s eta 0:00:00


In [39]:
%%writefile app.py

import streamlit as st
import pandas as pd

st.set_page_config(page_title="Data Quality Analyzer", layout="wide")

st.title("📊 Data Quality Report Generator")

file = st.file_uploader("Upload CSV", type=["csv"])

if file:
    df = pd.read_csv(file)

    st.subheader("Preview")
    st.dataframe(df.head())


    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
    if "joining_date" in df.columns:
        df["joining_date"] = pd.to_datetime(df["joining_date"], errors="coerce")


    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    duplicates = df.duplicated().sum()


    numeric_df = df.select_dtypes(include=['number'])

    Q1 = numeric_df.quantile(0.25)
    Q3 = numeric_df.quantile(0.75)
    IQR = Q3 - Q1

    outliers = ((numeric_df < (Q1 - 1.5 * IQR)) | (numeric_df > (Q3 + 1.5 * IQR)))
    outlier_count = outliers.sum()


    score = 100
    score -= int(missing_percent.mean())
    score -= duplicates
    score -= int(outlier_count.sum() / 10)
    score = max(0, score)

    st.subheader("Quality Score")
    st.metric("Score", f"{score}/100")


    grades = {}
    for col in df.columns:
        miss = missing_percent[col]
        if miss == 0:
            grades[col] = "A"
        elif miss < 10:
            grades[col] = "B"
        elif miss < 30:
            grades[col] = "C"
        else:
            grades[col] = "D"

    st.subheader("Column Grades")
    st.write(grades)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Missing (%)")
        st.write(missing_percent)

    with col2:
        st.subheader("Duplicates")
        st.write(duplicates)


    st.subheader("⚠️ Outliers per Column")
    st.write(outlier_count)

    st.subheader("📊 Outlier Visualization")
    st.bar_chart(outlier_count)


    st.subheader("Suggestions")
    if missing.sum() > 0:
        st.write("- Fill missing values")
    if duplicates > 0:
        st.write("- Remove duplicates")

    st.write("- Fix data types")
    st.write("- Handle outliers")


    clean_df = df.copy()
    clean_df = clean_df.drop_duplicates()
    clean_df = clean_df.fillna(clean_df.median(numeric_only=True))

    st.subheader("Cleaned Data")
    st.dataframe(clean_df.head())

    csv = clean_df.to_csv(index=False).encode('utf-8')
    st.download_button("Download Cleaned Data", csv, "cleaned.csv", "text/csv")

Overwriting app.py


In [40]:
from pyngrok import ngrok


ngrok.kill()


public_url = ngrok.connect(8501)
print("👉 Open this link:", public_url)

!streamlit run app.py &>/dev/null&

👉 Open this link: NgrokTunnel: "https://quartered-aloof-trimness.ngrok-free.dev" -> "http://localhost:8501"


In [41]:
from pyngrok import ngrok


ngrok.set_auth_token("3DDU2ytdiKKyUVG4tOy6sVzHS7a_5x5ThS2DAp8DX6tLbH6Kh")


ngrok.kill()


public_url = ngrok.connect(8501)
print("👉 Open this link:", public_url)


!streamlit run app.py &>/dev/null&

👉 Open this link: NgrokTunnel: "https://quartered-aloof-trimness.ngrok-free.dev" -> "http://localhost:8501"
